### **Student Information**
GitHub ID: TsungMin-Chang

Kaggle name: 115062518_張宗旻_Sandra

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

---

## 1. Model Development

### 1.1 Preprocessing Steps

- **Merging the data:** joined 3 files on `post_id`:
  - `final_posts.json` → the post text
  - `data_identification.csv` → train/test split
  - `emotion.csv` → labels
  - Result: **3,360 train / 1,440 test**, 6 emotions.
- **Light cleaning only.** Pretrained transformers already handle casing, emoji and slang, so heavy cleaning would throw away signal:
  - URLs → `http`
  - 4+ repeated letters → 3 ("soooooo" → "sooo"), which keeps the emphasis but cuts down on rare tokens
  - extra whitespace squeezed
- **On purpose, no** lowercasing, stopword removal or emoji stripping. "I'm FINE 🙃" and "i'm fine" don't feel the same, and emoji often *are* the emotion.

### 1.2 Feature Engineering Steps

- **For a transformer, the tokenizer is the feature engineering.** Each model uses its own subword tokenizer, max 128 tokens, padded per batch.
- **No hand-made features.** The transformer learns its own representation, so adding features by hand would mostly add noise.
- **Class weights in the loss.** Rarer emotions count more, so the model can't just ignore them. This matters because the metric is macro-F1, where every class counts equally.
- **Baseline for comparison:** TF-IDF word 1–2-grams + character 2–5-grams → LinearSVC. CV ≈ 0.51, public 0.5213. That gave me a floor to beat.

### 1.3 Explanation of Your Model

- **Approach:** fine-tuned pretrained transformers with **5-fold stratified CV**. Each fold predicts the test set, and the 5 predictions are averaged.
- **Training setup:** AdamW, lr 1e-5 (large) / 2e-5 (base), 5 epochs (best epoch kept), label smoothing 0.05, bf16, one RTX 3090.
- **Final model:** an average of 7 runs:
  - `j-hartmann/emotion-english-roberta-large` × 3 seeds
  - `SamLowe/roberta-base-go_emotions` × 3 seeds
  - plain `roberta-large` × 1
- **Why it works:**
  - The emotion-pretrained models start out already "knowing" emotions, so they need less of our small dataset to learn the task.
  - Averaging seeds smooths out lucky and unlucky runs, so one bad initialization can't drag the result down.
- **Results** (full table at the end of this notebook):
  - Single models: 0.605–0.629 CV → ensemble: **0.6435 CV**
  - Public LB 0.6394 (2-model submission), **private LB 0.6546, rank 18**
- **Tried and dropped:** DeBERTa-v3-large failed in both bf16 and fp32: the bf16 run produced invalid numbers (NaN), and the fp32 retry got stuck predicting a single class. So I removed it instead of spending more time debugging it.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude as a coding partner and tutor. Main prompts and what I decided:

| # | My prompt | What I decided |
|---|---|---|
| 1 | Asked it to get the competition part done quickly | Got a working submission first (the TF-IDF baseline), then went back to understand each step |
| 2 | "Be more specific on next steps, take it step by step." | Switched to small steps I could check one at a time, instead of big code dumps |
| 3 | Asked whether the leaderboard score counts toward the grade, and whether to keep pushing or move on | It counts: 5 pts for passing F1 0.5, plus up to 10 pts by private rank. But scores near my rank were packed within ~0.01, so more tuning was worth at most 1–2 pts. I did one "free" overnight run, then stopped and spent the time on Parts 1–2 (worth 70 pts) |
| 4 | "Don't we want to run something overnight, just for the sake of trying?" | My idea: ran the multi-seed ensemble overnight → **+0.012**, which became the final model |
| 5 | Debugging: Kaggle `id` column error, CUDA crash, DeBERTa failure | `id`: Kaggle rejected `post_id` because the sample file's header was wrong, so I renamed the column to `id`. CUDA: a random GPU crash wiped a run, so I saved each finished fold to disk and added automatic retries (up to 3), which means a crash only costs one fold. DeBERTa: tried bf16 and fp32, it failed both times, so I dropped it |

**Takeaway:** the AI was fastest at writing code and debugging. The decisions were mine: when to stop tuning, when to drop DeBERTa, and when to just try the overnight run.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
# Imports and the six competition labels
import argparse, os, re, time, random
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
L2I = {l: i for i, l in enumerate(LABELS)}


In [ ]:
# ---------- 1.Preprocessing ----------
def load_data(data_dir):
    """Join the three competition files into one table with text, split and label."""
    posts = pd.read_json(os.path.join(data_dir, "final_posts.json"))
    split = pd.read_csv(os.path.join(data_dir, "data_identification.csv"))
    labels = pd.read_csv(os.path.join(data_dir, "emotion.csv"))
    df = posts.merge(split, on="post_id").merge(labels, on="post_id", how="left")
    train = df[df.set == "train"].reset_index(drop=True)
    test = df[df.set == "test"].reset_index(drop=True)
    return train, test


def clean(text):
    """Light cleaning only: pretrained models already understand casing, emoji and slang."""
    text = re.sub(r"http\S+", "http", text)          # URLs carry no emotion
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)     # 'soooooo' -> 'sooo' (keeps the emphasis)
    return re.sub(r"\s+", " ", text).strip()

# Load the data and apply the light cleaning
DATA_DIR = "./data"   # folder with final_posts.json, data_identification.csv, emotion.csv
train, test = load_data(DATA_DIR)
X, y = train.text.map(clean).values, train.label.map(L2I).values
X_te = test.text.map(clean).values
print(train.shape, test.shape)
print(train.label.value_counts())

## 2. Feature Engineering Steps

In [ ]:
# ---------- 2. Feature engineering: tokenization ----------
class PostDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.enc = tok(list(texts), truncation=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = int(self.labels[i])
        return item


def make_loader(ds, tok, bs, shuffle):
    def collate(batch):
        labels = [b.pop("labels") for b in batch] if "labels" in batch[0] else None
        out = tok.pad(batch, return_tensors="pt")      # pad each batch to its own longest post
        if labels is not None:
            out["labels"] = torch.tensor(labels)
        return out
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate)


## 3. Model Implementation Steps

In [ ]:
# ---------- 3. Model: fine-tune and predict ----------
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), -1).cpu().numpy())
    return np.concatenate(probs)


def train_fold(args, tok, tr_x, tr_y, va_x, va_y, te_x, device, fold):
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model, num_labels=len(LABELS), ignore_mismatched_sizes=True,  # new 6-class head
        id2label=dict(enumerate(LABELS)), label2id=L2I).to(device)
    tr_dl = make_loader(PostDataset(tr_x, tr_y, tok, args.max_len), tok, args.bs, True)
    va_dl = make_loader(PostDataset(va_x, None, tok, args.max_len), tok, args.bs * 2, False)
    te_dl = make_loader(PostDataset(te_x, None, tok, args.max_len), tok, args.bs * 2, False)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)
    steps = len(tr_dl) * args.epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    # Class weights: rarer emotions (fear) count more, which helps macro-F1
    counts = np.bincount(tr_y, minlength=len(LABELS))
    weights = torch.tensor(counts.sum() / (len(LABELS) * counts), dtype=torch.float, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    best_f1, best_va, best_te = -1, None, None
    for ep in range(args.epochs):
        model.train()
        t0 = time.time()
        for batch in tr_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        va_p = predict(model, va_dl, device)
        f1 = f1_score(va_y, va_p.argmax(1), average="macro")
        print(f"  fold {fold} epoch {ep+1}: val macro-F1 {f1:.4f} ({time.time()-t0:.0f}s)", flush=True)
        if f1 > best_f1:   # keep the best epoch's predictions (early stopping)
            best_f1, best_va, best_te = f1, va_p, predict(model, te_dl, device)
    del model; torch.cuda.empty_cache()
    return best_f1, best_va, best_te


In [ ]:
# 5-fold cross-validation for each model in the final ensemble.
# (The same code ran as train_transformer.py on the lab GPU; see run_all.sh / overnight.sh.)
# Final ensemble = 7 runs: 2 emotion-pretrained models x 3 random seeds, plus plain roberta-large.
# (DeBERTa-v3-large was also tried, but it collapsed to predicting one class, so it was dropped.)
MODELS = [  # (Hugging Face model, learning rate, batch size, seed)
    ("j-hartmann/emotion-english-roberta-large", 1e-5, 16, 42),
    ("j-hartmann/emotion-english-roberta-large", 1e-5, 16, 7),
    ("j-hartmann/emotion-english-roberta-large", 1e-5, 16, 13),
    ("SamLowe/roberta-base-go_emotions",         2e-5, 32, 42),
    ("SamLowe/roberta-base-go_emotions",         2e-5, 32, 7),
    ("SamLowe/roberta-base-go_emotions",         2e-5, 32, 13),
    ("roberta-large",                            1e-5, 16, 42),
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_oof, all_te = [], []
for name, lr, bs, seed in MODELS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)   # a different seed = a different training run
    args = argparse.Namespace(model=name, lr=lr, bs=bs, epochs=5, max_len=128)
    tok = AutoTokenizer.from_pretrained(name)
    oof = np.zeros((len(X), len(LABELS))); te = np.zeros((len(X_te), len(LABELS)))
    # The fold split stays fixed (random_state=42) so every model is scored on the same held-out posts
    for fold, (tr_i, va_i) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y)):
        f1, va_p, te_p = train_fold(args, tok, X[tr_i], y[tr_i], X[va_i], y[va_i], X_te, device, fold)
        oof[va_i] = va_p; te += te_p / 5
    print(f"{name} (seed {seed}) OOF macro-F1:", round(f1_score(y, oof.argmax(1), average="macro"), 4))
    all_oof.append(oof); all_te.append(te)


In [ ]:
# Ensemble: average the probabilities of all models, then pick the most likely emotion
oof_ens, te_ens = np.mean(all_oof, 0), np.mean(all_te, 0)
print("Ensemble OOF macro-F1:", round(f1_score(y, oof_ens.argmax(1), average="macro"), 4))
# Kaggle expects the ID column to be named "id" (the sample file says post_id, which Kaggle rejects)
submission = pd.DataFrame({"id": test.post_id, "label": [LABELS[i] for i in te_ens.argmax(1)]})
submission.to_csv("submission.csv", index=False)
submission.head()

### Results from the lab-GPU run
The cells above were run as `train_transformer.py` / `ensemble.py` on one RTX 3090 (outputs copied from `train.log` and `overnight.log`). OOF = out-of-fold macro-F1 over all 3,360 labeled posts (5-fold CV).

| Model | Seed | OOF macro-F1 |
|---|---|---|
| j-hartmann/emotion-english-roberta-large | 42 | 0.6270 |
| j-hartmann/emotion-english-roberta-large | 7 | 0.6254 |
| j-hartmann/emotion-english-roberta-large | 13 | 0.6292 |
| SamLowe/roberta-base-go_emotions | 42 | 0.6219 |
| SamLowe/roberta-base-go_emotions | 7 | 0.6244 |
| SamLowe/roberta-base-go_emotions | 13 | 0.6209 |
| roberta-large | 42 | 0.6052 |
| **Ensemble (mean of 7)** | | **0.6435** |
| *(earlier 2-model ensemble: seed-42 j-hartmann + SamLowe)* | | *0.6315* |
| *(baseline: TF-IDF word+char n-grams + LinearSVC)* | | *0.513* |

Kaggle public leaderboard: TF-IDF baseline 0.5213, 2-model ensemble 0.6394. Final selected submissions: the 7-model and 2-model ensembles.

**Private leaderboard (final): rank 18, macro-F1 0.6546** (see the screenshot at the top of this notebook).
